# SAFB Minimum Verification (One-Click Kaggle)

这个 Notebook 只做最小化验证：
- `fixed_freqfed` vs `anb_freqfed`
- 输出 ACC / ASR / Bypass 和 hypothesis check


In [ ]:
from pathlib import Path
import os

candidates = [
    Path('/kaggle/working/SAFB'),
    Path('/kaggle/input/safb/SAFB'),
    Path('/kaggle/input/safb-project/SAFB'),
    Path('/kaggle/input/safb'),
]

project_root = None
for c in candidates:
    if (c / 'minimum verification' / 'run_minimum_verification.py').exists():
        project_root = c
        break

if project_root is None:
    for base in [Path('/kaggle/working'), Path('/kaggle/input')]:
        for p in base.rglob('run_minimum_verification.py'):
            if p.parent.name == 'minimum verification':
                project_root = p.parent.parent
                break
        if project_root is not None:
            break

if project_root is None:
    raise FileNotFoundError('未找到 SAFB 项目根目录，请先上传/挂载项目文件')

os.chdir(project_root)
print('PROJECT_ROOT =', project_root)


In [ ]:
import subprocess, sys

req = str(project_root / 'requirements.txt')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])
print('Dependencies installed from', req)


In [ ]:
import subprocess, sys

script = project_root / 'minimum verification' / 'run_minimum_verification.py'
# IMPORTANT: /kaggle/input is read-only, so outputs must go to /kaggle/working
output_dir = Path('/kaggle/working/results/minimum_verification')
output_dir.mkdir(parents=True, exist_ok=True)
data_dir = project_root / 'data'

cmd = [
    sys.executable, str(script),
    '--data-dir', str(data_dir),
    '--output-dir', str(output_dir),
    '--num-rounds', '30',
    '--train-subset', '6000',
    '--test-subset', '1200',
    '--poison-rate', '0.9',
    '--scaling-factor', '4.5',
    '--local-epochs', '3',
    '--seeds', '42', '123', '2026'
]

print('Running command:\n', ' '.join(cmd))
subprocess.check_call(cmd)


In [ ]:
import json
from pathlib import Path

summary_path = Path('/kaggle/working/results/minimum_verification/minimum_verification_summary.json')
print('Summary file:', summary_path)

with summary_path.open('r', encoding='utf-8') as f:
    payload = json.load(f)

print('\nHypothesis check:')
print(json.dumps(payload.get('summary', {}).get('hypothesis_check', {}), ensure_ascii=False, indent=2))

print('\nCondition summary rows:')
for row in payload.get('summary', {}).get('summary_rows', []):
    print(row)
